In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

#Step 1: Fault Labels

In [ ]:
canopy_2015 = pd.read_csv(
    "/content/drive/MyDrive/NIST_Preprocessed/canopy_2015_Labeled.csv"
)
canopy_2016 = pd.read_csv(
    "/content/drive/MyDrive/NIST_Preprocessed/canopy_2016_Labeled.csv"
)

In [ ]:
# Convert the TIMESTAMP column to pandas datetime object
canopy_2015["TIMESTAMP"] = pd.to_datetime(canopy_2015["TIMESTAMP"])
canopy_2016["TIMESTAMP"] = pd.to_datetime(canopy_2016["TIMESTAMP"])

# Set TIMESTAMP as index
canopy_2015.set_index("TIMESTAMP", inplace=True)
canopy_2016.set_index("TIMESTAMP", inplace=True)

In [ ]:
# Fault Indicator Columns
# Rather than merging label_string/label_inverter into a single priority-ordered
# column (which would hide genuine co-occurrence between fault types), we keep
# both label sets fully separate and add lightweight boolean indicators on top.
# This supports both a coarse "fault vs no-fault" detection task and full
# subsystem-level classification, without discarding information.

NON_FAULT_VALUES = ["NORMAL", "NOT_EVALUABLE", "MISSING_DATA", "INSUFFICIENT_DATA"]

canopy_2015["has_string_fault"] = ~canopy_2015["label_string"].isin(NON_FAULT_VALUES)
canopy_2015["has_inverter_fault"] = ~canopy_2015["label_inverter"].isin(
    NON_FAULT_VALUES
)
canopy_2015["has_any_fault"] = (
    canopy_2015["has_string_fault"] | canopy_2015["has_inverter_fault"]
)
canopy_2015["fault_cooccurrence"] = (
    canopy_2015["has_string_fault"] & canopy_2015["has_inverter_fault"]
)

# Summary
print("=== Fault Indicator Summary (2015) ===\n")
print(f"has_string_fault   : {canopy_2015['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2015['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2015['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2015['fault_cooccurrence'].sum()}")

print(
    f"\nAs % of evaluable rows (Op_Mask=1): "
    f"{canopy_2015.loc[canopy_2015['Op_Mask'] == 1, 'has_any_fault'].mean() * 100:.2f}%"
)

In [ ]:
NON_FAULT_VALUES = ["NORMAL", "NOT_EVALUABLE", "MISSING_DATA", "INSUFFICIENT_DATA"]

canopy_2016["has_string_fault"] = ~canopy_2016["label_string"].isin(NON_FAULT_VALUES)
canopy_2016["has_inverter_fault"] = ~canopy_2016["label_inverter"].isin(
    NON_FAULT_VALUES
)
canopy_2016["has_any_fault"] = (
    canopy_2016["has_string_fault"] | canopy_2016["has_inverter_fault"]
)
canopy_2016["fault_cooccurrence"] = (
    canopy_2016["has_string_fault"] & canopy_2016["has_inverter_fault"]
)

# Summary
print("=== Fault Indicator Summary (2016) ===\n")
print(f"has_string_fault   : {canopy_2016['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2016['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2016['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2016['fault_cooccurrence'].sum()}")

print(
    f"\nAs % of evaluable rows (Op_Mask=1): "
    f"{canopy_2016.loc[canopy_2016['Op_Mask'] == 1, 'has_any_fault'].mean() * 100:.2f}%"
)

In [ ]:
columns = ["Op_Mask", "InvPAC_kW_Avg", "has_any_fault"]
canopy_2015[columns].to_csv(
    "/content/drive/MyDrive/CS_LSTM/canopy_2015_normal_hour_analysis.csv", index=True
)

#Step 2: Build Healthy Mask

In [ ]:
def create_healthy_mask(df):

    return (df["Op_Mask"] == 1) & (df["InvOpStatus_Avg"] > 0) & (~df["has_any_fault"])


healthy_mask_2015 = create_healthy_mask(canopy_2015)
healthy_mask_2016 = create_healthy_mask(canopy_2016)

In [ ]:
print(f"Healthy Operating Periods (2015): {healthy_mask_2015.sum()}")
print(f"Healthy Operating Periods (2016): {healthy_mask_2016.sum()}")

#Step 3: Segment Analysis

In [ ]:
def analyze_segments(mask, label, seq_len=60):

    groups = (mask != mask.shift()).cumsum()

    segment_lengths = mask.groupby(groups).sum()

    segment_lengths = segment_lengths[segment_lengths > 0]

    usable = (segment_lengths - seq_len + 1).clip(lower=0)

    print(f"\n========== {label} ==========")
    print(f"Healthy timestamps            : {segment_lengths.sum():,}")
    print(f"Continuous healthy segments   : {len(segment_lengths):,}")
    print(f"Longest segment               : {segment_lengths.max():,}")
    print(f"Average segment               : {segment_lengths.mean():.1f}")
    print(f"Segments < {seq_len} samples  : {(usable == 0).sum():,}")
    print(f"Usable windows (L={seq_len})  : {usable.sum():,}")

    return segment_lengths

In [ ]:
segments_2015 = analyze_segments(healthy_mask_2015, "2015")
segments_2016 = analyze_segments(healthy_mask_2016, "2016")

In [ ]:
# Feature Selection & Contextual Encoding

PRIMARY_FEATURES = [
    "POA_fused",
    "AmbTemp_C_Avg",
    "module_temp",
    "InvVDCin_Avg",
    "InvIDCin_Avg",
    "InvPDC_kW_Avg",
    "I_string_mean",
    "InvPAC_kW_Avg_lag1",
]

CONTEXTUAL_FEATURES = [
    "sin_doy",
    "cos_doy",
    "sin_hour",
    "cos_hour",
]

TARGET = "InvPAC_kW_Avg"


def add_features(df):
    df = df.copy()

    # Autoregressive feature
    df["InvPAC_kW_Avg_lag1"] = df["InvPAC_kW_Avg"].shift(1)

    # Contextual features
    if "TIMESTAMP" in df.columns:
        df = df.set_index(pd.to_datetime(df["TIMESTAMP"]))

    doy = df.index.day_of_year.values
    hour = df.index.hour.values + df.index.minute.values / 60.0

    df["sin_doy"] = np.sin(2 * np.pi * doy / 365.0)
    df["cos_doy"] = np.cos(2 * np.pi * doy / 365.0)
    df["sin_hour"] = np.sin(2 * np.pi * hour / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * hour / 24.0)

    return df

In [ ]:
canopy_2015 = add_features(canopy_2015)
canopy_2016 = add_features(canopy_2016)

#Step 4: Extract Continuous Healthy Segments

In [ ]:
def extract_segments(df, healthy_mask):

    groups = (healthy_mask != healthy_mask.shift()).cumsum()

    segments = []

    for _, segment in df.groupby(groups):
        if not healthy_mask.loc[segment.index[0]]:
            continue

        segments.append(segment.copy())

    return segments

In [ ]:
segments2015 = extract_segments(canopy_2015, healthy_mask_2015)

segments2016 = extract_segments(canopy_2016, healthy_mask_2016)

In [ ]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

#Stage 5: Remove first fow of every segment

In [ ]:
def clean_segments(segments):

    cleaned = []

    removed = 0

    for seg in segments:
        if len(seg) <= 1:
            continue

        cleaned.append(seg.iloc[1:].copy())
        removed += 1

    print(f"Removed first row from {removed:,} segments.")

    return cleaned

In [ ]:
segments2015 = clean_segments(segments2015)
segments2016 = clean_segments(segments2016)

#Stage 6: Remove short segments

In [ ]:
SEQ_LEN = 60

segments2015 = [s for s in segments2015 if len(s) >= SEQ_LEN]

segments2016 = [s for s in segments2016 if len(s) >= SEQ_LEN]

In [ ]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

#Results

In [ ]:
print("=" * 60)
print("Healthy Segment Summary")
print("=" * 60)

print("2015:")
print(f"  Total healthy segments : {len(segments2015):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2015):,}")

print()

print("2016:")
print(f"  Total healthy segments : {len(segments2016):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2016):,}")

In [ ]:
import pandas as pd

lengths2015 = pd.Series([len(s) for s in segments2015])
lengths2016 = pd.Series([len(s) for s in segments2016])

print("========== 2015 ==========")
print(lengths2015.describe())

print("\n========== 2016 ==========")
print(lengths2016.describe())

In [ ]:
lengths2015 = pd.Series([len(s) for s in segments2015])

print("Top 10 longest healthy segments (2015):")

display(lengths2015.sort_values(ascending=False).head(10).rename("Length").to_frame())

In [ ]:
segment_id = 300

segment = segments2015[segment_id]

print(f"Segment {segment_id}")
print(f"Rows      : {len(segment)}")
print(f"Start     : {segment.index[0]}")
print(f"End       : {segment.index[-1]}")

display(segment)

#Windowing

In [ ]:
SEQ_LEN = 60  # 60 minutes
HORIZON = 1  # Predict next timestep

print(f"Sequence length : {SEQ_LEN}")
print(f"Prediction horizon : {HORIZON}")

In [ ]:
def create_windows_from_segment(
    segment, primary_features, contextual_features, target, seq_len=60, horizon=1
):
    """
    Generate CS-LSTM windows from one continuous healthy segment.

    Returns
    -------
    X_primary : (N, seq_len, n_primary)
    X_context : (N, seq_len, n_context)
    y         : (N,)
    """

    X_primary = []
    X_context = []
    y = []

    n = len(segment)

    for start in range(n - seq_len - horizon + 1):
        end = start + seq_len

        X_primary.append(segment.iloc[start:end][primary_features].values)

        X_context.append(segment.iloc[start:end][contextual_features].values)

        y.append(segment.iloc[end + horizon - 1][target])

    return X_primary, X_context, y

In [ ]:
def create_training_dataset(
    segments, primary_features, contextual_features, target, seq_len=60, horizon=1
):

    Xp = []
    Xc = []
    y = []

    total_segments = len(segments)
    skipped = 0

    for segment in segments:
        # Remove the first row (lag belongs to previous segment)
        segment = segment.iloc[1:]

        if len(segment) < seq_len + horizon:
            skipped += 1
            continue

        xp, xc, yy = create_windows_from_segment(
            segment, primary_features, contextual_features, target, seq_len, horizon
        )

        Xp.extend(xp)
        Xc.extend(xc)
        y.extend(yy)

    Xp = np.asarray(Xp, dtype=np.float32)
    Xc = np.asarray(Xc, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    print("=" * 60)
    print("Sliding Window Summary")
    print("=" * 60)
    print(f"Healthy segments            : {total_segments:,}")
    print(f"Skipped (<{seq_len + 1} rows) : {skipped:,}")
    print(f"Used segments               : {total_segments - skipped:,}")
    print(f"Generated windows           : {len(y):,}")
    print()

    print(f"Primary tensor   : {Xp.shape}")
    print(f"Context tensor   : {Xc.shape}")
    print(f"Target vector    : {y.shape}")

    return Xp, Xc, y

In [ ]:
Xp2015, Xc2015, y2015 = create_training_dataset(
    segments2015,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON,
)

In [ ]:
Xp2016, Xc2016, y2016 = create_training_dataset(
    segments2016,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON,
)

In [ ]:
Xp_train = np.concatenate([Xp2015, Xp2016], axis=0)
Xc_train = np.concatenate([Xc2015, Xc2016], axis=0)
y_train = np.concatenate([y2015, y2016], axis=0)

print("=" * 60)
print("Final Training Dataset")
print("=" * 60)

print(f"Primary tensor : {Xp_train.shape}")
print(f"Context tensor : {Xc_train.shape}")
print(f"Targets        : {y_train.shape}")